# 05 · Machine Learning Longitudinal — OASIS-2

## Pregunta de investigación

**¿Las trayectorias longitudinales de los pacientes permiten diferenciar a los sujetos `Converted` de los `Nondemented`?**

El análisis baseline utilizó exclusivamente información de la primera visita. En este notebook incorporamos información de la evolución posterior de cada paciente.

Por tanto, este análisis es **retrospectivo**: las variables de cambio anual necesitan conocer la primera y la última visita y no estaban disponibles en baseline. No deben interpretarse como variables de predicción precoz.

> **Datos:** el dataset original OASIS-2 no se distribuye con este repositorio. Se conservan código, métricas agregadas e interpretaciones, evitando publicar registros individuales o identificadores de participantes.


## ¿Qué cambia respecto al Machine Learning baseline?

Además de variables disponibles en la primera visita, incorporamos dos medidas que resumen la evolución de cada paciente:

| Variable | Significado |
|---|---|
| `MMSE_change_year` | Cambio anualizado de MMSE entre la primera y la última visita. Un valor negativo indica disminución de la puntuación cognitiva. |
| `nWBV_change_year` | Cambio anualizado del volumen cerebral normalizado entre la primera y la última visita. Un valor negativo indica disminución de nWBV. |
| `followup_years` | Duración del seguimiento en años. Se utiliza para calcular las tasas anualizadas, pero no como predictor. |

### Dos preguntas diferentes

**Baseline ML:** ¿la información disponible al inicio contiene señal sobre la futura conversión?

**Longitudinal ML:** una vez observada la evolución, ¿las trayectorias permiten discriminar retrospectivamente entre Converted y Nondemented?

Esta distinción evita confundir clasificación longitudinal con predicción precoz.


## 1. Importación de librerías y carga de datos


In [3]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [4]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

In [5]:
df_oasis2 = pd.read_excel("../data/oasis_2.xlsx")

## 2. Selección de la cohorte longitudinal

Se conservan únicamente los sujetos `Nondemented` y `Converted`. Los pacientes que ya pertenecen al grupo `Demented` no forman parte de esta comparación.

OASIS-2 contiene varias visitas por persona, por lo que inicialmente cada sujeto puede aparecer en varias filas.


In [9]:
df_longitudinal = df_oasis2[
    (df_oasis2['Group'] == 'Nondemented') |
    (df_oasis2['Group'] == 'Converted')
]

In [11]:
df_longitudinal.shape

(227, 15)

In [13]:
df_longitudinal['Group'].value_counts()

Group
Nondemented    190
Converted       37
Name: count, dtype: int64

In [15]:
df_longitudinal['Subject ID'].nunique()

86

La cohorte contiene **86 pacientes únicos: 72 Nondemented y 14 Converted**. Es la misma población utilizada en el análisis baseline, pero aquí aprovechamos también sus visitas posteriores.

El fuerte desbalance continúa siendo una limitación importante.


## 3. Construcción de una trayectoria por paciente

Las visitas se ordenan por paciente y número de visita. Después se identifican la primera y la última observación disponible de cada sujeto.

De esta forma podremos transformar un dataset con múltiples visitas por paciente en una tabla final con **una fila por persona**.


In [17]:
df_orden = df_longitudinal.sort_values(['Subject ID', 'Visit']) #Primero ordéname los registros por paciente y, 
# dentro de cada paciente, ordéname sus visitas de la primera a la última.

In [19]:
df_primera = df_orden.groupby('Subject ID').first()

In [21]:
df_ultima = df_orden.groupby('Subject ID').last()

In [23]:
df_primera.shape

(86, 14)

In [25]:
df_ultima.shape

(86, 14)

In [27]:
df_longitudinal_ml= df_primera[[
    'Group', 'M/F', 'Age', 'EDUC', 'SES', 'MMSE', 'nWBV'
]].copy()

### Duración del seguimiento

`followup_years` representa aproximadamente los años transcurridos entre baseline y la última visita disponible.

Es necesario porque los pacientes no fueron seguidos exactamente durante el mismo tiempo. Utilizaremos esta duración para expresar los cambios de MMSE y nWBV como tasas anuales.


In [29]:
df_longitudinal_ml['followup_years'] = df_ultima['MR Delay'] / 365

### Cambio anual de MMSE

MMSE (*Mini-Mental State Examination*) es una prueba cognitiva de 0 a 30 puntos. En términos generales, puntuaciones mayores representan mejor rendimiento cognitivo, aunque MMSE por sí solo no diagnostica demencia.

Calculamos:

**Cambio anual de MMSE = (MMSE final − MMSE inicial) / años de seguimiento**

- valor positivo → la puntuación aumentó;
- valor próximo a 0 → permaneció aproximadamente estable;
- valor negativo → la puntuación disminuyó.

MMSE puede fluctuar entre evaluaciones, por lo que esta tasa resume la trayectoria y no representa una medida perfecta del deterioro.


In [31]:
df_longitudinal_ml['MMSE_change_year'] = (df_ultima['MMSE']-df_primera['MMSE'])/df_longitudinal_ml['followup_years']

### Cambio anual de nWBV

`nWBV` (*Normalized Whole Brain Volume*) representa el volumen cerebral total normalizado teniendo en cuenta el tamaño intracraneal.

Calculamos:

**Cambio anual de nWBV = (nWBV final − nWBV inicial) / años de seguimiento**

Un valor negativo representa una disminución del volumen cerebral normalizado durante el seguimiento.


In [33]:
df_longitudinal_ml['nWBV_change_year'] = (df_ultima['nWBV']-df_primera['nWBV'])/df_longitudinal_ml['followup_years']

## 4. Dataset final para Machine Learning

La tabla final contiene una observación por paciente. Antes de modelar comprobamos el tamaño, los tipos de datos, los valores ausentes y las estadísticas agregadas de las variables longitudinales.


In [35]:
df_longitudinal_ml.shape

(86, 10)

In [37]:
df_longitudinal_ml.info()

<class 'pandas.core.frame.DataFrame'>
Index: 86 entries, OAS2_0001 to OAS2_0186
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Group             86 non-null     object 
 1   M/F               86 non-null     object 
 2   Age               86 non-null     int64  
 3   EDUC              86 non-null     int64  
 4   SES               86 non-null     float64
 5   MMSE              86 non-null     float64
 6   nWBV              86 non-null     float64
 7   followup_years    86 non-null     float64
 8   MMSE_change_year  86 non-null     float64
 9   nWBV_change_year  86 non-null     float64
dtypes: float64(6), int64(2), object(2)
memory usage: 7.4+ KB


In [39]:
df_longitudinal_ml.isnull().sum()

Group               0
M/F                 0
Age                 0
EDUC                0
SES                 0
MMSE                0
nWBV                0
followup_years      0
MMSE_change_year    0
nWBV_change_year    0
dtype: int64

In [41]:
df_longitudinal_ml.groupby('Group')[
    ['MMSE_change_year', 'nWBV_change_year']
].agg(['mean', 'median', 'std'])

MMSE_change_year                    nWBV_change_year            \
                        mean   median       std             mean    median   
Group                                                                        
Converted          -0.591398 -0.41126  0.859219        -0.005681 -0.005618   
Nondemented        -0.034680  0.00000  0.621386        -0.003580 -0.003875   

                       
                  std  
Group                  
Converted    0.004481  
Nondemented  0.004121

El EDA longitudinal mostró que el cambio anual de MMSE presenta diferencias más claras entre Converted y Nondemented que el cambio anual de nWBV. Ahora evaluaremos si estas trayectorias aportan capacidad discriminativa cuando se combinan con las variables basales.


## 5. Variables predictoras y preprocesamiento

El modelo utiliza:

**Variables baseline:** sexo, edad, educación, SES, MMSE y nWBV.

**Variables longitudinales:** cambio anual de MMSE y cambio anual de nWBV.

La variable objetivo se codifica como:

- `0 = Nondemented`
- `1 = Converted`

`followup_years` se utiliza para calcular las tasas de cambio, pero se excluye deliberadamente de los predictores porque la duración del seguimiento puede reflejar características del diseño del estudio.


In [43]:
X_longitudinal = df_longitudinal_ml[['M/F','Age','EDUC','SES','MMSE','nWBV', 'MMSE_change_year', 'nWBV_change_year']]

In [45]:
y = df_longitudinal_ml['Group'].map({
    'Nondemented': 0,
    'Converted':1
})

In [47]:
numeric_features = ['Age','EDUC','SES','MMSE','nWBV','MMSE_change_year','nWBV_change_year']

In [49]:
categorical_features = ['M/F']

In [51]:
scaler = StandardScaler()

In [53]:
encoder = OneHotEncoder()

In [55]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', scaler,numeric_features),
        ('cat', encoder,categorical_features)
    ]
)

## 6. Modelo y estrategia de validación

Se utiliza un **SVM lineal (Support Vector Machine)**. Este algoritmo intenta encontrar una frontera que separe lo mejor posible las dos clases.

Comparamos dos configuraciones:

- **SVM estándar**
- **SVM balanceado**, que penaliza más los errores cometidos sobre la clase minoritaria Converted.

Se utiliza la misma estrategia del análisis baseline: **Repeated Stratified 5-Fold Cross-Validation**, con 5 folds repetidos 10 veces.

La estratificación ayuda a mantener aproximadamente la proporción de Nondemented y Converted en cada partición. Las repeticiones permiten observar la estabilidad del rendimiento con diferentes divisiones de esta pequeña muestra.


In [57]:
# modelo balanceado SVM
svm_model_longitudinal_balanced = SVC(
    kernel='linear',
    probability=True,
    random_state=42,
    class_weight='balanced'
)

In [59]:
# modelo SVM
svm_model_longitudinal = SVC(
    kernel='linear',
    probability=True,
    random_state=42
)

In [61]:
pipeline_svm_longitudinal_balanced = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', svm_model_longitudinal_balanced)
])

In [63]:
pipeline_svm_longitudinal = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', svm_model_longitudinal)
])

In [65]:
cv_repeated = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

## 7. Métricas de evaluación

Como la cohorte está muy desbalanceada, no utilizamos únicamente accuracy:

- **Accuracy:** proporción total de clasificaciones correctas.
- **Recall:** proporción de Converted reales que detecta el modelo.
- **Precision:** de los pacientes clasificados como Converted, cuántos realmente lo son.
- **F1-score:** equilibrio entre precision y recall.
- **ROC-AUC:** capacidad global de separación entre ambas clases.
- **PR-AUC:** resume la relación Precision–Recall y resulta especialmente útil cuando la clase positiva es poco frecuente.


In [67]:
results_svm_longitudinal = cross_validate(
    pipeline_svm_longitudinal,
    X_longitudinal,
    y,
    cv=cv_repeated,
    scoring={
        'accuracy': 'accuracy',
        'recall': 'recall',
        'precision': 'precision',
        'f1': 'f1',
        'roc_auc': 'roc_auc',
        'pr_auc': 'average_precision'
    }
)

C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_clas

In [69]:
results_svm_longitudinal_balanced = cross_validate(
    pipeline_svm_longitudinal_balanced,
    X_longitudinal,
    y,
    cv=cv_repeated,
    scoring={
        'accuracy': 'accuracy',
        'recall': 'recall',
        'precision': 'precision',
        'f1': 'f1',
        'roc_auc': 'roc_auc',
        'pr_auc': 'average_precision'
    }
)

C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [71]:
metricas = [
    'accuracy',
    'recall',
    'precision',
    'f1',
    'roc_auc',
    'pr_auc'
]

In [73]:
for metrica in metricas:
    print(metrica)
    print(results_svm_longitudinal[f'test_{metrica}'].mean())
    print(results_svm_longitudinal[f'test_{metrica}'].std())
    print(results_svm_longitudinal_balanced[f'test_{metrica}'].mean())
    print(results_svm_longitudinal_balanced[f'test_{metrica}'].std())

accuracy
0.8198039215686274
0.03510607130498719
0.6739215686274511
0.12216329710950297
recall
0.04333333333333333
0.13666666666666666
0.5466666666666666
0.3073904502239601
precision
0.04666666666666666
0.14157840387730203
0.2606782106782107
0.14404310078854357
f1
0.0440952380952381
0.13493866919621092
0.34151692751692747
0.17776367266102103
roc_auc
0.6825079365079365
0.17093948161382033
0.6645079365079364
0.20901013512153496
pr_auc
0.42067253008429484
0.1999681920784149
0.4609811163781752
0.2423472578686965


## 8. Resultados

| Modelo | Accuracy | Recall | Precision | F1 | ROC-AUC | PR-AUC |
|---|---:|---:|---:|---:|---:|---:|
| SVM estándar | ≈ 0,82 | ≈ 0,04 | ≈ 0,05 | ≈ 0,04 | ≈ 0,68 | ≈ 0,42 |
| SVM balanceado | ≈ 0,67 | ≈ 0,55 | ≈ 0,26 | ≈ 0,34 | ≈ 0,67 | ≈ 0,46 |

Estas cifras representan las medias obtenidas en las 50 evaluaciones de validación cruzada y presentan una variabilidad considerable.

### Interpretación

El SVM estándar alcanza una accuracy cercana al **82 %**, pero detecta aproximadamente solo un **4 % de los Converted**. La cifra de accuracy es engañosa porque el modelo favorece fuertemente la clase mayoritaria.

Con `class_weight='balanced'`, el recall aumenta hasta aproximadamente **55 %**. Sin embargo, la precision permanece alrededor del **26 %**, lo que implica numerosos falsos positivos.

Existe cierta señal discriminativa, pero el rendimiento continúa siendo moderado y poco estable.


## 9. Comparación con el Machine Learning baseline

En el análisis baseline, una de las configuraciones relativamente más favorables fue el SVM balanceado sin CDR ni MMSE:

| Modelo | Recall | Precision | F1 | ROC-AUC | PR-AUC |
|---|---:|---:|---:|---:|---:|
| Baseline SVM balanceado | ≈ 0,61 | ≈ 0,23 | ≈ 0,33 | ≈ 0,65 | ≈ 0,42 |
| Longitudinal SVM balanceado | ≈ 0,55 | ≈ 0,26 | ≈ 0,34 | ≈ 0,67 | ≈ 0,46 |

La incorporación de información longitudinal **no produjo una mejora drástica**.

Esta comparación debe interpretarse únicamente de forma descriptiva. Los dos modelos no utilizan exactamente el mismo conjunto de variables: el longitudinal incluye MMSE basal y variables de cambio, mientras que el modelo baseline mostrado excluía CDR y MMSE. Por tanto, estas cifras no demuestran que un enfoque sea superior al otro.


## 10. Significación estadística frente a capacidad de clasificación

En el EDA longitudinal, el cambio anual de MMSE mostró diferencias más claras entre Converted y Nondemented que el cambio anual de nWBV.

Sin embargo, encontrar una diferencia estadística entre dos grupos **no significa automáticamente que podamos clasificar correctamente a cada individuo**.

Los pacientes continúan presentando solapamiento, solo disponemos de 14 Converted y existe variabilidad entre sus trayectorias. El modelo debe enfrentarse a todos estos factores simultáneamente.

Esta diferencia entre **significación estadística** y **capacidad predictiva/discriminativa** es una de las conclusiones metodológicas importantes del proyecto.


# Limitaciones del Machine Learning longitudinal

1. **Muestra pequeña y desbalanceada.** Solo 14 de los 86 pacientes pertenecen a Converted.
2. **Resumen mediante primera y última visita.** Las tasas no aprovechan completamente las visitas intermedias ni sus fluctuaciones.
3. **Duración de seguimiento variable.** La anualización facilita la comparación, pero no elimina todas las diferencias entre trayectorias.
4. **Información longitudinal limitada.** Solo se resumen MMSE y nWBV.
5. **Elevada variabilidad de las métricas.** El reducido número de Converted hace que el rendimiento dependa considerablemente de las particiones.
6. **Clasificación retrospectiva, no predicción precoz.** Las variables de cambio utilizan información posterior a baseline.
7. **Sin validación externa.** El análisis se realiza exclusivamente con OASIS-2.


# Conclusión del Machine Learning longitudinal

La información longitudinal analizada **no permitió obtener una separación robusta entre Converted y Nondemented**.

Aunque el cambio anual de MMSE mostró diferencias más claras entre ambos grupos que el cambio anual de nWBV, el rendimiento del modelo continuó siendo moderado y presentó una elevada variabilidad.

El balanceo permitió detectar más Converted, pero a costa de numerosos falsos positivos. Por tanto, las variables longitudinales disponibles y el reducido tamaño de la muestra son insuficientes para realizar una clasificación individual fiable.


# Conclusión general del proyecto

Este proyecto comenzó con una pregunta ambiciosa:

**¿Podemos predecir el Alzheimer antes de la aparición de los síntomas utilizando Machine Learning?**

El estudio de OASIS-2 obligó a reformular esa pregunta. Este dataset permite estudiar diferencias entre sujetos Nondemented, Demented y Converted y analizar su evolución, pero **no permite demostrar una predicción presintomática de Alzheimer a cinco años**.

El EDA baseline mostró que los pacientes que ya presentan demencia son relativamente distinguibles mediante variables cognitivas y estructurales. Sin embargo, los futuros Converted presentan un importante solapamiento con Nondemented en la primera visita.

Los modelos clásicos de Machine Learning confirmaron esa dificultad. El balanceo permitió aumentar la detección de Converted, pero la precision permaneció baja y las métricas mostraron una elevada variabilidad. La incorporación de cambios longitudinales de MMSE y nWBV tampoco produjo una separación claramente más robusta.

**Con la muestra y las variables estructuradas de OASIS-2 utilizadas en este proyecto, los modelos clásicos evaluados no muestran una capacidad suficientemente robusta y estable para predecir qué pacientes desarrollarán posteriormente demencia.**

Este resultado también es informativo: en problemas clínicos complejos, aumentar la sofisticación del algoritmo no sustituye a disponer de una cohorte adecuada, suficientes casos positivos y variables capaces de capturar la biología de la enfermedad.

Como trabajo futuro podrían estudiarse cohortes mayores y fuentes multimodales —por ejemplo imágenes MRI, biomarcadores, genética u otras medidas clínicas— utilizando validación externa rigurosa.
